# NSE Divergence Strategy Stock Analyzer

Interactive NSE equity chart with adjustable candle periods, corporate-action handling, and RSI/price divergence markers.

**Run order:** install the requirements cell, choose an NSE ticker in the next cell, then run the chart cell. The notebook downloads data from NSE and needs an internet connection.

> For research and educational use only; this is not investment advice. Market data availability, corporate-action classifications, and exchange policies can change.


In [ ]:
%pip install -r requirements.txt

In [ ]:
SYMBOL = "GESHIP".strip().upper()  # Change this ticker, then run the notebook.

In [ ]:
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from IPython.display import HTML, display

from datetime import date
from jugaad_data.nse import stock_df
from jugaad_data.nse import NSELive
import math
import re

if "SYMBOL" not in globals() or not SYMBOL.strip():
    raise ValueError("Set SYMBOL in the ticker cell before running the chart cell.")
SYMBOL = SYMBOL.strip().upper()
nse = NSELive()
quote = nse.stock_quote(SYMBOL)
if not isinstance(quote, dict) or not isinstance(quote.get("secInfo"), dict):
    raise ValueError(f"NSE returned invalid security metadata for {SYMBOL}. Check the ticker symbol.")
security_info = quote.get("secInfo", {})
listing_date_value = security_info.get("listingDate")
if not listing_date_value:
    raise ValueError(f"NSE did not provide a listing date for {SYMBOL}. Check the ticker symbol.")
listing_date = pd.to_datetime(listing_date_value, format="%d-%b-%Y %H:%M:%S").date()
today = date.today()
if listing_date > today:
    raise ValueError(f"NSE returned a future listing date for {SYMBOL}: {listing_date}.")

df = stock_df(symbol=SYMBOL, from_date=listing_date, to_date=today, series="EQ")
if df.empty:
    raise ValueError(f"NSE returned no daily price history for {SYMBOL} from its listing date ({listing_date}).")

chart = df.copy()
chart["DATE"] = pd.to_datetime(chart["DATE"])
if chart["DATE"].dt.hour.eq(18).all() and chart["DATE"].dt.minute.eq(30).all():
    chart["DATE"] += pd.Timedelta(hours=5, minutes=30)
chart["DATE"] = chart["DATE"].dt.normalize()
chart = chart.dropna(
    subset=["OPEN", "HIGH", "LOW", "CLOSE"]
).sort_values("DATE")
chart = chart.loc[chart["DATE"].dt.date >= listing_date].reset_index(drop=True)
if chart.empty:
    raise ValueError(f"No valid NSE daily candles found for {SYMBOL} on or after its listing date ({listing_date}).")

actions_url = "https://www.nseindia.com/api/corporates-corporateActions"
corporate_actions = []
for year in range(listing_date.year, today.year + 1):
    period_start = max(listing_date, date(year, 1, 1))
    period_end = min(today, date(year, 12, 31))
    response = nse.s.get(
        actions_url,
        params={
            "index": "equities", "symbol": SYMBOL,
            "from_date": period_start.strftime("%d-%m-%Y"),
            "to_date": period_end.strftime("%d-%m-%Y"),
        },
        timeout=30,
    )
    response.raise_for_status()
    period_actions = response.json()
    if not isinstance(period_actions, list):
        raise ValueError(f"Unexpected NSE corporate-action response for {SYMBOL} in {year}.")
    corporate_actions.extend(period_actions)
unique_actions = {}
for action in corporate_actions:
    action_key = (action.get("exDate"), action.get("subject"), action.get("symbol"))
    unique_actions[action_key] = action
corporate_actions = list(unique_actions.values())

split_adjustments = []
for action in corporate_actions:
    subject = str(action.get("subject", ""))
    if "split" not in subject.lower() and "sub-division" not in subject.lower():
        continue
    face_values = re.search(
        r"from\s+(?:rs\.?|re\.?)\s*([\d,.]+).*?to\s+(?:rs\.?|re\.?)\s*([\d,.]+)",
        subject, flags=re.IGNORECASE,
    )
    ex_date = pd.to_datetime(action.get("exDate"), format="%d-%b-%Y", errors="coerce")
    if not face_values or pd.isna(ex_date):
        raise ValueError(f"Could not read the split ratio or ex-date from NSE action: {subject}")
    old_face, new_face = (float(value.replace(",", "")) for value in face_values.groups())
    if old_face <= 0 or new_face <= 0:
        raise ValueError(f"Invalid split face values in NSE action: {subject}")
    split_adjustments.append((ex_date.normalize(), new_face / old_face))

adjustment_events = []
other_price_sensitive_actions = []
for action in corporate_actions:
    subject = str(action.get("subject", ""))
    subject_lower = subject.lower()
    dividend_per_share = None
    dividend_percent = None
    face_value = None
    if "split" in subject_lower or "sub-division" in subject_lower:
        face_values = re.search(
            r"from\s+(?:rs\.?|re\.?)\s*([\d,.]+).*?to\s+(?:rs\.?|re\.?)\s*([\d,.]+)",
            subject, flags=re.IGNORECASE,
        )
        if not face_values:
            raise ValueError(f"Could not read the split ratio from NSE action: {subject}")
        old_face, new_face = (float(value.replace(",", "")) for value in face_values.groups())
        if old_face <= 0 or new_face <= 0:
            raise ValueError(f"Invalid split face values in NSE action: {subject}")
        price_factor = new_face / old_face
        adjust_share_counts = True
    elif "bonus" in subject_lower:
        ratio = re.search(r"bonus.*?(\d+)\s*:\s*(\d+)", subject, flags=re.IGNORECASE)
        if not ratio:
            raise ValueError(f"Could not read the bonus ratio from NSE action: {subject}")
        bonus_shares, existing_shares = (int(value) for value in ratio.groups())
        if bonus_shares <= 0 or existing_shares <= 0:
            raise ValueError(f"Invalid bonus ratio in NSE action: {subject}")
        price_factor = existing_shares / (existing_shares + bonus_shares)
        adjust_share_counts = True
    elif "dividend" in subject_lower:
        dividend = re.search(r"(?:rs\.?|re\.?)\s*([\d,.]+)\s*(?:per\s+share)?", subject, flags=re.IGNORECASE)
        if not dividend:
            dividend_percent = re.search(r"([\d,.]+)\s*%", subject)
            face_value = pd.to_numeric(action.get("faceVal"), errors="coerce")
            if not dividend_percent or pd.isna(face_value) or face_value <= 0:
                raise ValueError(f"Could not read the cash dividend per share from NSE action: {subject}")
            dividend_percent = float(dividend_percent.group(1).replace(",", ""))
        else:
            dividend_per_share = float(dividend.group(1).replace(",", ""))
        adjust_share_counts = False
    elif any(term in subject_lower for term in ("rights", "demerger", "amalgamation", "merger", "capital reduction")):
        other_price_sensitive_actions.append(subject)
        continue
    else:
        continue

    ex_date_value = action.get("exDate")
    ex_date = pd.to_datetime(ex_date_value, format="%d-%b-%Y", errors="coerce")
    if pd.isna(ex_date):
        raise ValueError(f"Missing or invalid ex-date for NSE corporate action: {subject}")
    ex_date = ex_date.normalize()
    if "dividend" in subject_lower:
        if dividend_per_share is None:
            later_split_ratio = math.prod(
                1 / split_factor for split_date, split_factor in split_adjustments
                if split_date > ex_date
            )
            dividend_per_share = dividend_percent * face_value * later_split_ratio / 100
        previous_candles = chart.loc[chart["DATE"] < ex_date, "CLOSE"]
        if previous_candles.empty:
            raise ValueError(f"No prior closing price is available to adjust dividend action: {subject}")
        previous_close = previous_candles.iloc[-1]
        if dividend_per_share <= 0 or dividend_per_share >= previous_close:
            raise ValueError(f"Dividend amount is invalid relative to the prior close: {subject}")
        price_factor = (previous_close - dividend_per_share) / previous_close
    adjustment_events.append((ex_date, price_factor, adjust_share_counts, subject))

price_columns = ["OPEN", "HIGH", "LOW", "CLOSE", "PREV. CLOSE", "LTP", "VWAP"]
share_columns = ["VOLUME", "DELIVERY QTY"]
for ex_date, price_factor, adjust_share_counts, subject in sorted(adjustment_events, key=lambda event: event[0]):
    before_action = chart["DATE"] < ex_date
    for column in price_columns:
        if column in chart.columns:
            chart.loc[before_action, column] *= price_factor
    if adjust_share_counts:
        for column in share_columns:
            if column in chart.columns:
                chart.loc[before_action, column] /= price_factor

quote_metadata = quote.get("metaData")
company_name = quote_metadata.get("companyName", SYMBOL) if isinstance(quote_metadata, dict) else SYMBOL
print(f"{company_name} ({SYMBOL}) | listed {listing_date} | data through {chart['DATE'].max().date()}")
print(f"Loaded {len(chart):,} daily candles and applied {len(adjustment_events)} NSE split/bonus/dividend adjustment(s).")
if chart["DATE"].min().date() > listing_date:
    print(f"NSE history begins {chart['DATE'].min().date()}, after the listing date; earlier daily records are not available from this feed.")
if other_price_sensitive_actions:
    print("Review additional corporate actions that need event-specific adjustment:")
    for subject in other_price_sensitive_actions:
        print(f"  - {subject}")

def calculate_rsi(close):
    price_change = close.diff()
    average_gain = price_change.clip(lower=0).ewm(
        alpha=1 / 14, min_periods=14, adjust=False
    ).mean()
    average_loss = (-price_change.clip(upper=0)).ewm(
        alpha=1 / 14, min_periods=14, adjust=False
    ).mean()
    relative_strength = average_gain / average_loss
    rsi = 100 - (100 / (1 + relative_strength))
    rsi.loc[(average_loss == 0) & (average_gain > 0)] = 100
    rsi.loc[(average_gain == 0) & (average_loss > 0)] = 0
    rsi.loc[(average_gain == 0) & (average_loss == 0)] = 50
    return rsi

chart["RSI"] = calculate_rsi(chart["CLOSE"])


if chart.empty:
    raise ValueError("No usable price data found for the selected date range.")

timeframes = {
    "Daily": None,
    "Weekly": "W-FRI",
    "Monthly": "ME",
    "3 Months": "3ME",
    "6 Months": "6ME",
    "12 Months": "YE",
}
aggregation = {
    "DATE": "last", "OPEN": "first", "HIGH": "max", "LOW": "min",
    "CLOSE": "last", "VOLUME": "sum",
}

def aggregate_candles(frame, frequency):
    if frequency is None:
        candles = frame[list(aggregation)].copy()
        candles["RSI"] = calculate_rsi(candles["CLOSE"])
        return candles
    candles = frame.resample(frequency, on="DATE").agg(aggregation)
    candles = candles.dropna(subset=["OPEN", "HIGH", "LOW", "CLOSE"]).reset_index(drop=True)
    candles["RSI"] = calculate_rsi(candles["CLOSE"])
    return candles

def find_pivots(values, kind, window=2):
    values = pd.Series(values, dtype="float64").to_numpy()
    pivots = []
    for index in range(window, len(values) - window):
        neighborhood = values[index - window:index + window + 1]
        if not pd.notna(neighborhood).all():
            continue
        left = values[index - window:index]
        right = values[index + 1:index + window + 1]
        if kind == "high" and values[index] > left.max() and values[index] >= right.max():
            pivots.append(index)
        elif kind == "low" and values[index] < left.min() and values[index] <= right.min():
            pivots.append(index)
    return pivots

def find_divergences(candles, kind, window=2, matching_tolerance=2):
    price_column = "HIGH" if kind == "high" else "LOW"
    price_pivots = find_pivots(candles[price_column], kind, window)
    rsi_pivots = find_pivots(candles["RSI"], kind, window)
    candidates = sorted(
        (abs(price_index - rsi_index), price_index, rsi_index)
        for price_index in price_pivots
        for rsi_index in rsi_pivots
        if abs(price_index - rsi_index) <= matching_tolerance
    )
    used_price, used_rsi = set(), set()
    pairs = []
    for _, price_index, rsi_index in candidates:
        if price_index not in used_price and rsi_index not in used_rsi:
            pairs.append((price_index, rsi_index))
            used_price.add(price_index)
            used_rsi.add(rsi_index)
    pairs.sort()
    price_pivot_positions = {index: position for position, index in enumerate(price_pivots)}
    rsi_pivot_positions = {index: position for position, index in enumerate(rsi_pivots)}

    def divergence_continues(previous, current):
        if price_pivot_positions[current[0]] != price_pivot_positions[previous[0]] + 1:
            return False
        if rsi_pivot_positions[current[1]] != rsi_pivot_positions[previous[1]] + 1:
            return False
        previous_price = candles[price_column].iloc[previous[0]]
        current_price = candles[price_column].iloc[current[0]]
        previous_rsi = candles["RSI"].iloc[previous[1]]
        current_rsi = candles["RSI"].iloc[current[1]]
        if kind == "high":
            return current_price > previous_price and current_rsi < previous_rsi
        return current_price < previous_price and current_rsi > previous_rsi

    runs = []
    run = []
    for pair in pairs:
        if not run or divergence_continues(run[-1], pair):
            run.append(pair)
        else:
            if len(run) >= 3:
                runs.append(run)
            run = [pair]
    if len(run) >= 3:
        runs.append(run)
    return runs

def divergence_coordinates(runs, x_values, candles, kind, on_rsi):
    x_points, y_points, pivot_dates, point_labels = [], [], [], []
    for run in runs:
        for point_number, (price_index, rsi_index) in enumerate(run):
            index = rsi_index if on_rsi else price_index
            value_column = "RSI" if on_rsi else ("HIGH" if kind == "high" else "LOW")
            x_points.append(x_values[index])
            y_points.append(candles[value_column].iloc[index])
            pivot_dates.append(candles["DATE"].iloc[index].strftime("%b %d, %Y"))
            point_labels.append(
                f"{'Bearish' if kind == 'high' else 'Bullish'} divergence ({len(run)} {'peaks' if kind == 'high' else 'troughs'})"
                if point_number == len(run) - 1 else ""
            )
        x_points.append(None)
        y_points.append(None)
        pivot_dates.append("")
        point_labels.append("")
    return x_points, y_points, pivot_dates, point_labels

def year_window_slider(candles, x_values):
    last_date = candles["DATE"].iloc[-1]
    steps = []
    for years in range(1, 9):
        start_date = last_date - pd.DateOffset(years=years)
        start_index = int(candles["DATE"].searchsorted(start_date, side="left"))
        start_index = min(start_index, len(candles) - 1)
        start_x = x_values.iloc[start_index]
        steps.append({
            "label": f"{years}Y", "method": "relayout",
            "args": [{
                "xaxis.range": [start_x, x_values.iloc[-1]],
                "xaxis2.range": [start_x, x_values.iloc[-1]],
                "xaxis3.range": [start_x, x_values.iloc[-1]],
            }],
        })
    steps.append({
        "label": "All", "method": "relayout",
        "args": [{
            "xaxis.range": [x_values.iloc[0], x_values.iloc[-1]],
            "xaxis2.range": [x_values.iloc[0], x_values.iloc[-1]],
            "xaxis3.range": [x_values.iloc[0], x_values.iloc[-1]],
        }],
    })
    return steps

fig = make_subplots(
    rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.04,
    row_heights=[0.58, 0.25, 0.17],
)
timeframe_buttons = []
timeframe_sliders = []
traces_per_timeframe = 7
for timeframe_index, (timeframe, frequency) in enumerate(timeframes.items()):
    candles = aggregate_candles(chart, frequency)
    candle_count = len(candles)
    dates = candles["DATE"].dt.strftime("%b %d, %Y")
    x_values = candles["DATE"]
    visible = timeframe_index == 0
    timeframe_sliders.append(year_window_slider(candles, x_values))
    candle_spacing_ms = (
        candles["DATE"].diff().dropna().dt.total_seconds().median() * 1000
        if candle_count > 1 else 24 * 60 * 60 * 1000
    )

    fig.add_trace(go.Candlestick(
        x=x_values, open=candles["OPEN"], high=candles["HIGH"],
        low=candles["LOW"], close=candles["CLOSE"], name=SYMBOL,
        customdata=dates, visible=visible,
        increasing_line_color="#16a34a", decreasing_line_color="#dc2626",
        increasing_fillcolor="#16a34a", decreasing_fillcolor="#dc2626",
        hovertemplate=(
            f"{SYMBOL}<br>Date: %{{customdata}}<br>Open: ₹%{{open:.2f}}<br>"
            "High: ₹%{high:.2f}<br>Low: ₹%{low:.2f}<br>"
            "Close: ₹%{close:.2f}<extra></extra>"
        )
    ), row=1, col=1)

    volume_colors = [
        "#16a34a" if close >= open_ else "#dc2626"
        for open_, close in zip(candles["OPEN"], candles["CLOSE"])
    ]
    fig.add_trace(go.Bar(
        x=x_values, y=candles["VOLUME"], name="Volume",
        customdata=dates, visible=visible, marker_color=volume_colors,
        width=candle_spacing_ms * 0.7,
        hovertemplate="Date: %{customdata}<br>Volume: %{y:,.0f}<extra></extra>"
    ), row=2, col=1)

    fig.add_trace(go.Scatter(
        x=x_values, y=candles["RSI"], name="RSI (14)",
        customdata=dates, visible=visible, mode="lines",
        line={"color": "#7c3aed", "width": 1.5},
        hovertemplate="Date: %{customdata}<br>RSI: %{y:.2f}<extra></extra>"
    ), row=3, col=1)

    for kind, color, label in (
        ("high", "#ea580c", "Bearish divergence"),
        ("low", "#0891b2", "Bullish divergence"),
    ):
        runs = find_divergences(candles, kind)
        for on_rsi, row in ((False, 1), (True, 3)):
            x_points, y_points, pivot_dates, point_labels = divergence_coordinates(
                runs, x_values, candles, kind, on_rsi
            )
            fig.add_trace(go.Scatter(
                x=x_points, y=y_points, customdata=pivot_dates, text=point_labels,
                name=label, mode="lines+markers+text", visible=visible,
                line={"color": color, "width": 2, "dash": "dot"},
                marker={"color": color, "size": 9, "symbol": "diamond"},
                textposition="top center", textfont={"size": 10, "color": color},
                hovertemplate=f"{label}<br>Date: %{{customdata}}<br>Value: %{{y:.2f}}<extra></extra>",
            ), row=row, col=1)

    tick_count = min(candle_count, 9)
    tick_indices = sorted({
        round(i * (candle_count - 1) / max(tick_count - 1, 1))
        for i in range(tick_count)
    })
    tick_values = [x_values[i] for i in tick_indices]
    tick_labels = [candles["DATE"].iloc[i].strftime("%b %Y") for i in tick_indices]
    range_start, range_end = x_values.iloc[0], x_values.iloc[-1]
    trace_visibility = [False] * (len(timeframes) * traces_per_timeframe)
    trace_visibility[timeframe_index * traces_per_timeframe:(timeframe_index + 1) * traces_per_timeframe] = [True] * traces_per_timeframe
    timeframe_buttons.append({
        "label": timeframe,
        "method": "update",
        "args": [
            {"visible": trace_visibility},
            {
                "title.text": f"{company_name} (NSE: {SYMBOL}) — {timeframe} Candles",
                "xaxis.tickvals": tick_values, "xaxis.ticktext": tick_labels,
                "xaxis.range": [range_start, range_end],
                "xaxis2.range": [range_start, range_end],
                "xaxis3.tickvals": tick_values, "xaxis3.ticktext": tick_labels,
                "xaxis3.range": [range_start, range_end],
                "yaxis.autorange": True, "yaxis2.autorange": True,
                "yaxis3.range": [0, 100],
                "sliders": [{
                    "active": 8, "steps": timeframe_sliders[-1],
                    "currentvalue": {"prefix": "Window: ", "visible": True},
                    "x": 0.1, "y": -0.12, "len": 0.8,
                    "pad": {"t": 15, "b": 5},
                }],
            },
        ],
    })

fig.add_hline(y=70, line_dash="dot", line_color="gray", row=3, col=1)
fig.add_hline(y=30, line_dash="dot", line_color="gray", row=3, col=1)
daily_candles = aggregate_candles(chart, timeframes["Daily"])
daily_ticks = min(len(daily_candles), 9)
daily_tick_indices = sorted({
    round(i * (len(daily_candles) - 1) / max(daily_ticks - 1, 1))
    for i in range(daily_ticks)
})
daily_tick_values = [daily_candles["DATE"].iloc[i] for i in daily_tick_indices]
daily_tick_labels = [daily_candles["DATE"].iloc[i].strftime("%b %Y") for i in daily_tick_indices]
trading_days = pd.DatetimeIndex(chart["DATE"].dt.normalize().unique())
calendar_days = pd.date_range(trading_days.min(), trading_days.max(), freq="D")
non_trading_weekdays = calendar_days[
    (calendar_days.dayofweek < 5) & ~calendar_days.isin(trading_days)
]
date_rangebreaks = [
    {"bounds": ["sat", "mon"]},
    {"values": non_trading_weekdays.strftime("%Y-%m-%d").tolist()},
]

fig.update_layout(
    height=880, template="plotly_white", hovermode="x unified",
    dragmode="zoom", showlegend=False,
    title={"text": f"{company_name} (NSE: {SYMBOL}) — Daily Candles", "x": 0.5},
    margin={"l": 60, "r": 30, "t": 115, "b": 100},
    updatemenus=[{
        "type": "buttons", "direction": "right", "showactive": True,
        "x": 0, "y": 1.15, "xanchor": "left", "yanchor": "top",
        "buttons": timeframe_buttons,
    }],
    sliders=[{
        "active": 8, "steps": timeframe_sliders[0],
        "currentvalue": {"prefix": "Window: ", "visible": True},
        "x": 0.1, "y": -0.12, "len": 0.8, "pad": {"t": 15, "b": 5},
    }],
    xaxis_rangeslider_visible=False,
    xaxis2_rangeslider_visible=False,
    xaxis3_rangeslider_visible=False,
)
fig.update_yaxes(title_text="Price (INR)", row=1, col=1, fixedrange=False)
fig.update_yaxes(title_text="Volume", row=2, col=1, fixedrange=False)
fig.update_yaxes(title_text="RSI", range=[0, 100], row=3, col=1, fixedrange=False)
fig.update_xaxes(
    type="date",
    tickmode="array", tickvals=daily_tick_values, ticktext=daily_tick_labels,
    range=[daily_candles["DATE"].iloc[0], daily_candles["DATE"].iloc[-1]],
    rangebreaks=date_rangebreaks, showgrid=True,
    showspikes=False,
)
fig.update_xaxes(title_text="Date", row=3, col=1)

hover_script = """
const chart = document.getElementById('{plot_id}');
const guide = document.createElement('div');
guide.style.cssText = 'position:absolute;display:none;pointer-events:none;z-index:1;border-left:1px dotted #334155;';
chart.style.position = 'relative';
chart.appendChild(guide);
chart.addEventListener('pointermove', event => {
    const rect = chart.getBoundingClientRect();
    const {l, t, w, h} = chart._fullLayout._size;
    const x = event.clientX - rect.left;
    const y = event.clientY - rect.top;
    if (x < l || x > l + w || y < t || y > t + h) {
        guide.style.display = 'none';
        return;
    }
    guide.style.left = `${x}px`;
    guide.style.top = `${t}px`;
    guide.style.height = `${h}px`;
    guide.style.display = 'block';
});
chart.addEventListener('pointerleave', () => {
    guide.style.display = 'none';
});
"""
chart_html = pio.to_html(
    fig, full_html=False, include_plotlyjs="cdn", post_script=hover_script
)
display(HTML(chart_html))